In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\Etc\NLP\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [3]:
df.duplicated(subset=["review"]).sum()

418

In [4]:
df.drop_duplicates(subset=["review"], inplace=True)

In [5]:
df.sentiment.value_counts()

sentiment
positive    24884
negative    24698
Name: count, dtype: int64

## 1. Pretrained Embeddings (with fasttext)

In [6]:
df.review[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

#### convert output as "\__label__ text"

In [7]:
df.rename(columns={"sentiment":"label"}, inplace=True)

In [8]:
df["label"] = "__label__"+df['label'].astype(str)
df.head()

,review,label
0,One of the other reviewers has mentioned that ...,__label__positive
1,A wonderful little production. <br /><br />The...,__label__positive
2,I thought this was a wonderful way to spend ti...,__label__positive
3,Basically there's a family where a little boy ...,__label__negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",__label__positive


#### Preprocessing

In [9]:
import spacy 
from spacy.lang.en.stop_words import STOP_WORDS
nlp = spacy.load("en_core_web_sm")
import re



In [10]:
def preprocess(text):
    
    text = re.sub(r'<.*?>', ' ', text)

    text = re.sub(r'\s+', ' ', text)
    
    doc = nlp(text)

    filtered_token=[]
    for token in doc:
        if token.is_punct:
            continue
        
        if token.is_stop and token.text.lower() not in ["no", "not", "never"]:
            continue

        filtered_token.append(token.lemma_)

    return " ".join(filtered_token)


In [ ]:
df["clean_review"] = df["review"].map(preprocess)

In [ ]:
df["text"] = df["label"]+" "+df["clean_review"]
df.head()

,review,label,clean_review,text
0,One of the other reviewers has mentioned that ...,__label__positive,reviewer mention watch 1 Oz episode hook right...,__label__positive reviewer mention watch 1 Oz ...
1,A wonderful little production. <br /><br />The...,__label__positive,wonderful little production filming technique ...,__label__positive wonderful little production ...
2,I thought this was a wonderful way to spend ti...,__label__positive,think wonderful way spend time hot summer week...,__label__positive think wonderful way spend ti...
3,Basically there's a family where a little boy ...,__label__negative,basically family little boy Jake think zombie ...,__label__negative basically family little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",__label__positive,Petter Mattei love Time money visually stunnin...,__label__positive Petter Mattei love Time mone...


In [ ]:
df.head()

,review,label,clean_review,text
0,One of the other reviewers has mentioned that ...,__label__positive,reviewer mention watch 1 Oz episode hook right...,__label__positive reviewer mention watch 1 Oz ...
1,A wonderful little production. <br /><br />The...,__label__positive,wonderful little production filming technique ...,__label__positive wonderful little production ...
2,I thought this was a wonderful way to spend ti...,__label__positive,think wonderful way spend time hot summer week...,__label__positive think wonderful way spend ti...
3,Basically there's a family where a little boy ...,__label__negative,basically family little boy Jake think zombie ...,__label__negative basically family little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",__label__positive,Petter Mattei love Time money visually stunnin...,__label__positive Petter Mattei love Time mone...


#### Splitting the data

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2)

In [ ]:
train.shape, test.shape

((39665, 4), (9917, 4))

In [ ]:
train.head()

,review,label,clean_review,text
14557,"Before I'd seen this, I had seen some pretty b...",__label__negative,see see pretty bad Christmas film see Jingle w...,__label__negative see see pretty bad Christmas...
15739,OK heres what I say: <br /><br />The movie was...,__label__positive,ok here movie excellent huge Nancy fan read 1 ...,__label__positive ok here movie excellent huge...
38686,The first of five St Trinian's films (although...,__label__positive,St Trinian film usually discount base artist R...,__label__positive St Trinian film usually disc...
41773,Rock Hudson's second venture in the science fi...,__label__negative,Rock Hudson second venture science fiction gen...,__label__negative Rock Hudson second venture s...
43549,After 21 movies and three years of working in ...,__label__positive,21 movie year work Hollywood Bette Davis final...,__label__positive 21 movie year work Hollywood...


#### save training and test data in csv format

In [ ]:
train["text"].to_csv("sentiment.train",index=False, header=False)
test["text"].to_csv("sentiment.test",index=False, header=False)

#### Model training using fasttext

In [ ]:
import fasttext

model = fasttext.train_supervised(input="sentiment.train")

In [ ]:
model.test("sentiment.test")

(9492, 0.8960176991150443, 0.8960176991150443)

#### Prediction

In [ ]:
model.predict("laughed my heart out")

(('__label__positive',), array([0.96012521]))

In [ ]:
model.predict("waste of time")

(('__label__negative',), array([1.00001001]))

In [ ]:
model.predict("probably hear bit new Disney dub Miyazaki classic film Laputa Castle Sky late summer 1998 Disney release Kiki Delivery Service video include preview Laputa dub say 1999 obviously way past year dub finally complete not Laputa Castle Sky Castle sky dub Laputa not nice word Spanish use word Laputa time dub probably hear world renowned composer Joe Hisaishi score movie originally go rescore excellent music new arrangement Laputa come Neighbor Totoro Nausicaa Valley Wind begin Studio Ghibli long string hit opinion think Miyazaki good film powerful lesson tucker inside hour minute gem Laputa Castle Sky film age urge unfamiliar Castle Sky story begin right start stop hour storytelling flawless masterfully craft Miyazaki true vision believe fantastic film begin Sheeta girl helluva past hold captive government airship Sheeta hold key Laputa castle sky long lose civilization key Laputa sacred pendant seek government military air pirate group Dola gang Sheeta Pazu later befriend soon pirate attack ship escape raid fall thousand foot fall soft thank pendant float sky Pazu orphan boy survive work mine see Sheeta catch fast friend thank pendant catch huge thrill ride Dola gang government try capture Sheeta action sequence learn character motif identity build emotional action pack climax surely fantastic animation wonderful dialogue plus somewhat twisty surprise think film simply remarkable hold hour minute run time story wonderful peak Hayao Miyazaki animation no limit setting film combo time period place end 1800 alternante universe advance technology weapon Laputa surprisingly funny film film ton hilarious moment equal drama action film hold think funniest fight scene Pazu boss face pirate soon riot break funny man compare strength music fit right perfectly let talk dub rate excellent cast great performance bring character life teen heartthrob James Van Der Beek play hero Pazu mature voice japanese version original sound childlike way think voice nice fit Pazu Anna Paquin young Oscar winner Piano play Sheeta nice performance voice bit uneven stay true accent time sound american apple pie time sound like New Zealand performance enjoy Coris Leachman play Mama Dola not excellent performance voice emotion give character bring life live action laputa movie G d forbid play imagine role somewhat Luke Skywalker Mark Hamill Muska rate Hamill performance familiar Hamill long line voice work original Star Wars movie render Muska evil voice sound like regular voice mix Joker play episode animate Batman series round cast voice character actor Jim Cummings great gruff job general Andy Dick Mandy Patakin member Dola gang let talk make dub special Joe Hisaishi newly arrange music never hear Mr. Hisaishi music like Miyazaki film music memorable score persona fit particular film perfectly new arrangement american like think goal new recording worry classic tune japanese version great form score sound arrange like Hollywood blockbuster power emphasis clear deep film prologue second introduce airship new music not sure believe see ship no music majority music new backdrop background music enjoy thing enhanced powerful scene music strong original version calm scene calm overall think pleased new arrangement mix highly personally think help improve film prefer new score old hope Disney release license music right blown soundtrack plus dub story remain faithful original japanese line intact Kiki sure line change way line change majority close exactly original line dialogue Miyazaki write afraid excellent line butcher intact new line add help not sure consider good thing bad thing Disney not translate end song Japanese mortify completely new song Kiki dub version original song Japanese guess good original bad majority people see dub speak English big dub deal voice match character lip course dub will perfect think Kiki Mononoke dubbing line match well execute Disney little bit time time match perfect time completley match rare case say lip scene Sheeta chuckle mouth bit far thing film thought think amazing Laputa animation opening sequence ending animation lush detailed watch awe true nature character true detail face extreme close up action ton credit effort animator film beautifully hand draw like move piece art think mid 1980 animation different Disney Ghibli distinctive flare different good year color look vibrant Laputa ton action sequence lot plane dogfight plus ground sequence intriguing scary comparable big budget action film finale sound effect pure classic fit explosion gun fire like Miyazaki film focus different theme i.g kiki confidence great lesson greed power People realize greed have power good People obsess power greedy main villian Muska greatly show Laputa Castle Sky great film begin improve glad mainstream audience chance classic animate film glory great voice cast lot film excellent redone musical score Joe Hisaishi Disney nice job dub worthy think voice match mouth well Kiki Princess Mononoke Disney dub Castle Sky great dub worth long delay expierence fantastic film")


(('__label__positive',), array([0.99855578]))

In [ ]:


model.predict("movie fan never hear book Shirley Jackson Haunting Hill House never see 1963 Robert Wise production Julie Harris remake pretty darn bad plain awful bad acting Neeson think goofy computer enhancement away Jackson story doom remake favor rent original movie effectively scar hokey special effect acting professional believable reader book 1963 follow close")

(('__label__negative',), array([0.99585831]))

In [ ]:
model.predict("amazing one, not boring at all")

(('__label__negative',), array([0.89566594]))

In [ ]:
model.get_nearest_neighbors("laugh")

[(0.9921102523803711, 'perfect'),
 (0.9921067357063293, 'warm'),
 (0.9919740557670593, 'highly'),
 (0.9919219017028809, 'excellent'),
 (0.9917502403259277, 'gem'),
 (0.9916890263557434, 'extraordinary'),
 (0.9916723370552063, 'wonderfully'),
 (0.9916309118270874, 'Elvira'),
 (0.9916234016418457, 'unique'),
 (0.9916081428527832, 'outstanding')]

## 2. Traditional ML (Tf-idf + ML)
```
Text
 ↓
TF-IDF / Bag of Words
 ↓
Logistic Regression / Naive Bayes / Random Forest
```

In [ ]:
df.head()

,review,label,clean_review,text
0,One of the other reviewers has mentioned that ...,__label__positive,reviewer mention watch 1 Oz episode hook right...,__label__positive reviewer mention watch 1 Oz ...
1,A wonderful little production. <br /><br />The...,__label__positive,wonderful little production filming technique ...,__label__positive wonderful little production ...
2,I thought this was a wonderful way to spend ti...,__label__positive,think wonderful way spend time hot summer week...,__label__positive think wonderful way spend ti...
3,Basically there's a family where a little boy ...,__label__negative,basically family little boy Jake think zombie ...,__label__negative basically family little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",__label__positive,Petter Mattei love Time money visually stunnin...,__label__positive Petter Mattei love Time mone...


In [ ]:
target = {"__label__positive": 1, "__label__negative": 0}
df["label_num"]= df.label.map(target)
df.head()

,review,label,clean_review,text,label_num
0,One of the other reviewers has mentioned that ...,__label__positive,reviewer mention watch 1 Oz episode hook right...,__label__positive reviewer mention watch 1 Oz ...,1
1,A wonderful little production. <br /><br />The...,__label__positive,wonderful little production filming technique ...,__label__positive wonderful little production ...,1
2,I thought this was a wonderful way to spend ti...,__label__positive,think wonderful way spend time hot summer week...,__label__positive think wonderful way spend ti...,1
3,Basically there's a family where a little boy ...,__label__negative,basically family little boy Jake think zombie ...,__label__negative basically family little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",__label__positive,Petter Mattei love Time money visually stunnin...,__label__positive Petter Mattei love Time mone...,1


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(df.clean_review, df.label_num, test_size=0.2, random_state=2022, stratify=df.label_num)

In [ ]:
y_train.value_counts()

label_num
1    19907
0    19758
Name: count, dtype: int64

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [ ]:
from sklearn.naive_bayes import MultinomialNB


pipeline = Pipeline([("vectorizer", TfidfVectorizer()),
                     ("nb", MultinomialNB())
                    ])
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4940
           1       0.86      0.86      0.86      4977

    accuracy                           0.86      9917
   macro avg       0.86      0.86      0.86      9917
weighted avg       0.86      0.86      0.86      9917



In [ ]:
from sklearn.ensemble import RandomForestClassifier


pipeline = Pipeline([("vectorizer", TfidfVectorizer()),
                     ("nb", RandomForestClassifier())
                    ])
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85      4940
           1       0.85      0.85      0.85      4977

    accuracy                           0.85      9917
   macro avg       0.85      0.85      0.85      9917
weighted avg       0.85      0.85      0.85      9917



In [ ]:
from sklearn.linear_model import LogisticRegression


pipeline = Pipeline([("vectorizer", TfidfVectorizer()),
                     ("nb", LogisticRegression())
                    ])
pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.87      0.89      4940
           1       0.88      0.91      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



## 3. ANN with Tokenizer + Embedding

```
Text
 ↓
Tokenizer
 ↓
Integer Sequences
 ↓
Padding
 ↓
Embedding Layer
 ↓
ANN
```

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

`Tokenizer` is a class in TensorFlow/Keras.  (convert words -> integers)

num_words -> keep only the top most frequent words in vocabulary

#### Methods :
fit_on_texts(): build vocabulary
Tokenizer scans all reviews and assigns ids

texts_to_sequences(): convert words → integer ids
Ex: "movie was good"  -> [1, 45, 2]
Now text becomes numeric.

pad_sequences(): make all reviews same length
Because ANN requires fixed-size input.

In [ ]:
tokenizer = Tokenizer(num_words=50000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_review"])
# With OOV token: unknown word gets replaced with special token


In [ ]:
seq = tokenizer.texts_to_sequences(df["clean_review"])

In [ ]:
x = pad_sequences(seq, maxlen=200)

In [ ]:

x_train, x_test, y_train, y_test = train_test_split(x, df.label_num, test_size=0.2, random_state=2022, stratify=df.label_num)

Embedding converts: word ids → meaningful dense vectors

Example:
45 → [0.12, -0.88, ...]
18 → [0.55, 0.31, ...]

(without embedding it will treat the vectors as numeric values)

----------------------------------------------------------------------------------------------------
Embedding(input_dim, output_dim)

input_dim = vocabulary size (model can handle top 50,000 words)

output_dim = embedding vector size (each word becomes a vector of 128 numbers)


In [ ]:
model = Sequential([
    Embedding(input_dim=50000, output_dim=128),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(optimizer = 'adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, epochs=10, batch_size=32, validation_split=0.2)
# Batch 1  -> 32 samples -> update weights
# Batch 2  -> 32 samples -> update weights
# ...
# Batch 992 -> last 32 samples -> update weights


Epoch 1/10


992/992 ━━━━━━━━━━━━━━━━━━━━ 69s 68ms/step - accuracy: 0.7789 - loss: 0.4384 - val_accuracy: 0.8815 - val_loss: 0.2825
Epoch 2/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 68s 69ms/step - accuracy: 0.9505 - loss: 0.1490 - val_accuracy: 0.8688 - val_loss: 0.3087
Epoch 3/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 73s 73ms/step - accuracy: 0.9925 - loss: 0.0246 - val_accuracy: 0.8623 - val_loss: 0.6005
Epoch 4/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 71s 71ms/step - accuracy: 0.9966 - loss: 0.0103 - val_accuracy: 0.8393 - val_loss: 0.8195
Epoch 5/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 70s 71ms/step - accuracy: 0.9965 - loss: 0.0105 - val_accuracy: 0.8567 - val_loss: 0.9925
Epoch 6/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 69s 70ms/step - accuracy: 0.9973 - loss: 0.0091 - val_accuracy: 0.8505 - val_loss: 0.8549
Epoch 7/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 71s 71ms/step - accuracy: 0.9973 - loss: 0.0096 - val_accuracy: 0.8439 - val_loss: 0.7870
Epoch 8/10
992/992 ━━━━━━━━━━━━━━━━━━━━ 70s 70ms/step - accuracy: 0.9977 - loss: 0.0067 - val_accurac

In [ ]:
y_pred = model.predict(x_test)
y_pred = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred))

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
              precision    recall  f1-score   support

           0       0.87      0.84      0.85      4940
           1       0.84      0.88      0.86      4977

    accuracy                           0.86      9917
   macro avg       0.86      0.86      0.86      9917
weighted avg       0.86      0.86      0.86      9917

